In [ ]:
# 1. Install Unsloth
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 2. Install dependencies WITHOUT the xformers version limit!
!pip install -q --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
!pip install -U bitsandbytes>=0.46.1
print("Done")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"

# Configure 4-bit Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# Load the Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load the Model across both T4 GPUs
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# Prepare for training and inject LoRA Adapters
model = prepare_model_for_kbit_training(model)
peft_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)

In [ ]:
from datasets import load_dataset

# IMPORTANT: Replace these paths with your actual Kaggle dataset paths!
# You can find the exact paths by exploring the 'Input' folder on the right panel in Kaggle.
data_files = {
    "train": "/kaggle/input/datasets/naurosromim/bdshs/train.csv",
    "validation": "/kaggle/input/datasets/naurosromim/bdshs/val.csv",
    "test": "/kaggle/input/datasets/naurosromim/bdshs/test.csv"
}

# Load the dataset splits
dataset = load_dataset("csv", data_files=data_files)

In [ ]:
prompt_template = """You are an expert at analyzing Bengali social media text to detect hate speech.
Analyze the following Bengali text and extract three specific attributes.

Return your analysis STRICTLY in the following JSON format without any additional text:
{{
  "hate_presence": "Hate Speech" or "Not Hate Speech",
  "hate_type": "Call to Violence", "Slander", "Religious Hate", or "Not Specified",
  "target_category": "Individual", "Male", "Female", "Group", or "Not Specified"
}}

### Input Text:
{text}

### Response:
{response}"""

def format_dataset(example):
    # 1. Map the binary integer label to a clear string
    presence = "Hate Speech" if str(example["hate_presence"]) == "1" else "Not Hate Speech"

    # 2. Extract Type and Target. Using .get() ensures it won't crash if a row is missing data
    h_type = str(example.get("hate_type", "Not Specified"))
    target = str(example.get("target_category", "Not Specified"))

    # Logical override: If it's not hate speech, the type and target MUST be "Not Specified"
    if presence == "Not Hate Speech":
        h_type = "Not Specified"
        target = "Not Specified"

    # 3. Construct the expected JSON output
    expected_json = f'{{\n  "hate_presence": "{presence}",\n  "hate_type": "{h_type}",\n  "target_category": "{target}"\n}}'

    # 4. Combine into the final prompt and append the model's specific End-Of-Sequence token
    full_prompt = prompt_template.format(text=example["text"], response=expected_json) + tokenizer.eos_token

    return {"formatted_prompt": full_prompt}

In [ ]:
!pip install -q -U transformers accelerate

In [ ]:
!pip install -q -U trl peft accelerate bitsandbytes

In [ ]:
# Strip out the conflicting new versions
!pip uninstall -y transformers trl

# Install the exact stable versions that Unsloth requires
!pip install -q transformers==4.43.4 trl==0.8.6

In [ ]:
# Install Unsloth
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Install everything else WITHOUT version locks so they sync perfectly
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes transformers

In [ ]:
# Not Necessary
from transformers import TrainingArguments

# Set up the hyperparameter configurations
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,      # Small physical batch size to fit in VRAM
    gradient_accumulation_steps=4,      # 2 (batch) x 4 (accumulation) = 8 effective batch size
    learning_rate=2e-4,                 # Keeping your proven learning rate
    logging_steps=10,
    max_steps=100,                      # ⚠️ FOR TESTING: We will only run 100 steps first!
    # num_train_epochs=1,               # Uncomment this later for the full training run
    save_strategy="steps",
    save_steps=50,
    eval_strategy="steps",              # <-- CHANGED THIS LINE
    eval_steps=50,
    optim="paged_adamw_8bit",           # Crucial memory-saving optimizer
    fp16=True,                          # T4 GPUs use fp16, not bf16
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
)

In [ ]:
!pip install -q transformers==5.3.0 trl==0.24.0 datasets==4.3.0

In [ ]:
import os
print("Forcing Colab to completely reboot and clear the corrupted memory...")
os.kill(os.getpid(), 9)

In [ ]:
from trl import SFTTrainer

# Initialize the trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset["train"],
    eval_dataset=formatted_dataset["validation"],
    dataset_text_field="formatted_prompt", # The column we created in Phase 2
    max_seq_length=2048,                   # Matches your initial setup
    args=training_args,
    packing=False,                         # Keep false for structured JSON output
)

In [ ]:
import torch
import inspect
from trl import SFTTrainer

# 1. Dynamically Auto-Detect Configuration Standard
try:
    from trl import SFTConfig
    ConfigClass = SFTConfig
    is_modern = True
except ImportError:
    from transformers import TrainingArguments
    ConfigClass = TrainingArguments
    is_modern = False

# 2. Setup Core Configuration (Memory-safe for T4)
config_kwargs = {
    "output_dir": "/content/drive/MyDrive/DATASET/checkpoints",
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-4,
    "logging_steps": 10,
    "num_train_epochs": 1,
    "save_strategy": "steps",
    "save_steps": 100,
    "eval_strategy": "epoch",
    "optim": "adamw_8bit",
    "fp16": True,
    "bf16": False,
    "weight_decay": 0.01,
    "lr_scheduler_type": "linear",
    "seed": 3407,
}

# 3. Auto-Route Dataset Settings
if is_modern:
    config_kwargs["dataset_text_field"] = "formatted_prompt"
    config_kwargs["max_length"] = 256
    config_kwargs["packing"] = False

training_args = ConfigClass(**config_kwargs)

trainer_kwargs = {
    "model": model,
    "train_dataset": formatted_dataset["train"],
    "eval_dataset": formatted_dataset["validation"],
    "args": training_args,
}

if not is_modern:
    trainer_kwargs["dataset_text_field"] = "formatted_prompt"
    trainer_kwargs["max_seq_length"] = 256
    trainer_kwargs["packing"] = False

# 4. Auto-Detect Tokenizer/Processing Class naming
trainer_params = inspect.signature(SFTTrainer.__init__).parameters
if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

# 5. Initialize and Launch!
trainer = SFTTrainer(**trainer_kwargs)

torch.cuda.empty_cache()
print("🚀 Auto-Configuration successful! Launching the training loop...")
trainer.train()

# Save the finished LoRA adapters to Google Drive
drive_save_path = "/content/drive/MyDrive/DATASET/llama-3-8b-bd-shs-lora"
model.save_pretrained(drive_save_path)
tokenizer.save_pretrained(drive_save_path)
print(f"✅ Training Complete! Model weights successfully saved to: {drive_save_path}")

In [ ]:
import json
import torch
import re
import pandas as pd
from datasets import Dataset
from tqdm import tqdm
from sklearn.metrics import classification_report, accuracy_score

# =======================================================================
# ⚙️ 1. PIPELINE CONFIGURATION & DATA LOADING
# =======================================================================
print("🔄 Initializing Turbo Evaluation Pipeline...")

df_test = pd.read_csv("/kaggle/input/datasets/naurosromim/bdshs/test.csv")
test_data = Dataset.from_pandas(df_test)

# ⚡ SMART BATCHING: Sort by length to minimize padding waste
def compute_length(example):
    return {"text_length": len(str(example["sentence"]))}
    
test_data = test_data.map(compute_length).sort("text_length") 

# ⚡ CRITICAL: Left padding is required for decoder-only LLMs
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ⚡ INCREASED BATCH SIZE: 16 (or even 32 if your GPU memory allows)
BATCH_SIZE = 16 
results = []

print(f"🚀 Starting Turbo Generation on {len(test_data)} sentences with Batch Size {BATCH_SIZE}...")

# =======================================================================
# 🚀 2. INFERENCE & EXTRACTION ENGINE (SAFE BATCHING)
# =======================================================================
for i in tqdm(range(0, len(test_data), BATCH_SIZE), desc="Evaluating Batches"):
    batch = test_data[i:i+BATCH_SIZE]
    
    # --- A. CLEAN GROUND TRUTH ---
    true_presences, true_types, true_targets = [], [], []
    
    for j in range(len(batch["sentence"])):
        is_hate = str(batch["hate speech"][j]) == "1"
        t_presence = "Hate Speech" if is_hate else "Not Hate Speech"
        
        raw_type = str(batch.get("type", ["Not Specified"] * len(batch["sentence"]))[j]).lower()
        raw_target = str(batch.get("target", ["Not Specified"] * len(batch["sentence"]))[j]).lower()
        
        if not is_hate:
            t_type, t_target = "Not Specified", "Not Specified"
        else:
            if "call" in raw_type or "violence" in raw_type: t_type = "Call to Violence"
            elif "gender" in raw_type: t_type = "Gender-based Hate"
            elif "religion" in raw_type or "religious" in raw_type: t_type = "Religious Hate"
            elif "slander" in raw_type: t_type = "Slander"
            else: t_type = "Not Specified"
            
            if "female" in raw_target: t_target = "Female"
            elif "male" in raw_target: t_target = "Male"
            elif "group" in raw_target: t_target = "Group"
            elif "ind" in raw_target or "individual" in raw_target: t_target = "Individual"
            else: t_target = "Not Specified"
            
        true_presences.append(t_presence)
        true_types.append(t_type)
        true_targets.append(t_target)

    # --- B. LLM GENERATION ---
    prompts = [prompt_template.format(text=str(s), response="") for s in batch["sentence"]]
    
    # ⚡ Explicitly grab attention_mask
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    
    with torch.inference_mode():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"], # ⚡ THIS keeps batching safe!
            max_new_tokens=85,                       # Slightly reduced to save time
            use_cache=True, 
            do_sample=False,      
            pad_token_id=tokenizer.pad_token_id
        )
    
    # Slice off the prompt, keeping only the newly generated tokens
    generated_tokens = outputs[:, inputs["input_ids"].shape[1]:]
    generated_texts = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
    
    # --- C. REGEX PARSING & DATA LOGGING ---
    for j, text in enumerate(generated_texts):
        p_presence, p_type, p_target = "Parse Error", "Parse Error", "Parse Error"
        
        try:
            match = re.search(r'\{.*?\}', text, re.DOTALL)
            if match:
                pred_json = json.loads(match.group(0))
                p_presence = pred_json.get("hate_presence", "Parse Error")
                p_type = pred_json.get("hate_type", "Parse Error")
                p_target = pred_json.get("target_category", "Parse Error")
        except Exception:
            pass 
            
        results.append({
            "sentence": batch["sentence"][j],
            "true_presence": true_presences[j],
            "pred_presence": p_presence,
            "true_type": true_types[j],
            "pred_type": p_type,
            "true_target": true_targets[j],
            "pred_target": p_target,
            "raw_llm_output": text 
        })
            
    # Clean up memory instantly
    del inputs, outputs, generated_tokens
    torch.cuda.empty_cache()

# =======================================================================
# 📊 3. METRICS COMPUTATION & EXPORT
# =======================================================================
print("\n✅ Inference Complete! Computing Metrics...\n")

df_results = pd.DataFrame(results)
df_results.to_csv("hate_tiny_llm_evaluation_results.csv", index=False)
print("💾 Saved full predictions to 'hate_tiny_llm_evaluation_results.csv'")

def print_metrics(task_name, true_col, pred_col):
    print(f"\n{'='*40}")
    print(f"🏆 RESULTS: {task_name.upper()}")
    print(f"{'='*40}")
    
    valid_mask = df_results[pred_col] != "Parse Error"
    y_true = df_results[valid_mask][true_col]
    y_pred = df_results[valid_mask][pred_col]
    
    parse_errors = len(df_results) - len(y_true)
    if parse_errors > 0:
        print(f"⚠️ Warning: Ignored {parse_errors} sentences due to JSON Parse Errors.")
        
    acc = accuracy_score(y_true, y_pred)
    print(f"Overall Accuracy: {acc * 100:.2f}%")
    print("\nDetailed Classification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))

print_metrics("Hate Presence Detection", "true_presence", "pred_presence")
print_metrics("Hate Speech Type Extraction", "true_type", "pred_type")
print_metrics("Target Category Extraction", "true_target", "pred_target")

In [ ]:
import json
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# We will store the ground truth and the model's predictions in these lists
y_true_presence, y_pred_presence = [], []
y_true_type, y_pred_type = [], []
y_true_target, y_pred_target = [], []

In [ ]:
# 1. Install the Google Drive downloader
!pip install -q gdown

# 2. ⚠️ PASTE YOUR GOOGLE DRIVE FILE ID HERE
file_id = 'https://drive.google.com/drive/folders/1vOnj9uZtOtj1dbkO9agtJ0dMghyVP2T6?usp=drive_link' 

# 3. Download and Unzip!
print("Downloading weights from Google Drive...")
!gdown {file_id}
!unzip -q lora_weights.zip -d /kaggle/working/
print("✅ Weights successfully transferred to Kaggle!")

In [ ]:
from datasets import load_dataset

# ⚠️ IMPORTANT: Update this path to point to your actual test.csv file!
# If in Colab: "/content/drive/MyDrive/DATASET/bd-shs/test.csv"
# If in Kaggle: "/kaggle/input/YOUR_DATASET_NAME/test.csv"
test_file_path = "/kaggle/input/datasets/naurosromim/bdshs/test.csv"

# Load the dataset
dataset = load_dataset("csv", data_files={"test": test_file_path})

# Define the test_data variable that the next cell is looking for
test_data = dataset["test"]

print(f"✅ Test dataset loaded! Found {len(test_data)} sentences ready for evaluation.")

In [ ]:
import torch
import gc

# Force garbage collection and empty the GPU cache
gc.collect()
torch.cuda.empty_cache()
print("🧹 GPU memory successfully cleared!")

In [ ]:
import json
import torch
import re
import pandas as pd
from datasets import Dataset
from tqdm import tqdm
from sklearn.metrics import classification_report, accuracy_score

# =======================================================================
# ⚙️ 1. PIPELINE CONFIGURATION & DATA LOADING
# =======================================================================
print("🔄 Initializing Evaluation Pipeline...")

# Load dataset
df_test = pd.read_csv("/kaggle/input/datasets/naurosromim/bdshs/test.csv")
test_data = Dataset.from_pandas(df_test)
print(f"✅ Loaded {len(test_data)} test samples.")

# Smart Batching (Sort by length to minimize padding)
def compute_length(example):
    return {"text_length": len(str(example["sentence"]))}
    
test_data = test_data.map(compute_length).sort("text_length") 

# Tokenizer Setup
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

BATCH_SIZE = 8 

# Output Storage
results = []

# =======================================================================
# 🚀 2. INFERENCE & EXTRACTION ENGINE
# =======================================================================
print("🚀 Starting Generation Engine...")

for i in tqdm(range(0, len(test_data), BATCH_SIZE), desc="Evaluating Batches"):
    batch = test_data[i:i+BATCH_SIZE]
    
    # --- A. CLEAN GROUND TRUTH ---
    true_presences, true_types, true_targets = [], [], []
    
    for j in range(len(batch["sentence"])):
        # Presence
        is_hate = str(batch["hate speech"][j]) == "1"
        t_presence = "Hate Speech" if is_hate else "Not Hate Speech"
        
        # Type & Target (Defaulting to Non-Hate if not toxic)
        if not is_hate:
            t_type, t_target = "Not Specified", "Not Specified"
        else:
            # Type
            raw_type = str(batch.get("type", ["Not Specified"] * len(batch["sentence"]))[j]).lower()
            if "calltoviolence" in raw_type: t_type = "Call to Violence"
            elif "slander" in raw_type: t_type = "Slander"
            elif "religion" in raw_type: t_type = "Religious Hate"
            else: t_type = "Not Specified"
            
            # Target
            raw_target = str(batch.get("target", ["Not Specified"] * len(batch["sentence"]))[j]).lower()
            if "female" in raw_target: t_target = "Female"
            elif "male" in raw_target: t_target = "Male"
            elif "group" in raw_target: t_target = "Group"
            elif "ind" in raw_target: t_target = "Individual"
            else: t_target = "Not Specified"
            
        true_presences.append(t_presence)
        true_types.append(t_type)
        true_targets.append(t_target)

    # --- B. LLM GENERATION ---
    prompts = [prompt_template.format(text=str(s), response="") for s in batch["sentence"]]
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    
    with torch.inference_mode():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=100, 
            use_cache=True, 
            do_sample=False,      
            pad_token_id=tokenizer.pad_token_id
        )
    
    generated_tokens = outputs[:, inputs["input_ids"].shape[1]:]
    generated_texts = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
    
    # --- C. REGEX PARSING & DATA LOGGING ---
    for j, text in enumerate(generated_texts):
        p_presence, p_type, p_target = "Parse Error", "Parse Error", "Parse Error"
        
        try:
            match = re.search(r'\{.*?\}', text, re.DOTALL)
            if match:
                pred_json = json.loads(match.group(0))
                p_presence = pred_json.get("hate_presence", "Parse Error")
                p_type = pred_json.get("hate_type", "Parse Error")
                p_target = pred_json.get("target_category", "Parse Error")
        except Exception:
            pass # Keep defaults as "Parse Error"
            
        # Log everything for this specific sentence
        results.append({
            "sentence": batch["sentence"][j],
            "true_presence": true_presences[j],
            "pred_presence": p_presence,
            "true_type": true_types[j],
            "pred_type": p_type,
            "true_target": true_targets[j],
            "pred_target": p_target,
            "raw_llm_output": text # Saving raw text helps debug Parse Errors later!
        })
            
    del inputs, outputs, generated_tokens
    torch.cuda.empty_cache()

# =======================================================================
# 📊 3. METRICS COMPUTATION & EXPORT
# =======================================================================
print("\n✅ Inference Complete! Computing Metrics...\n")

# Convert results to a clean DataFrame
df_results = pd.DataFrame(results)

# Save to CSV for qualitative analysis and confusion matrices
df_results.to_csv("hate_tiny_llm_evaluation_results.csv", index=False)
print("💾 Saved full predictions to 'hate_tiny_llm_evaluation_results.csv'")

# Calculate and Print Academic Metrics
def print_metrics(task_name, true_col, pred_col):
    print(f"\n{'='*40}")
    print(f"🏆 RESULTS: {task_name.upper()}")
    print(f"{'='*40}")
    
    # Filter out parse errors for metric calculation
    valid_mask = df_results[pred_col] != "Parse Error"
    y_true = df_results[valid_mask][true_col]
    y_pred = df_results[valid_mask][pred_col]
    
    parse_errors = len(df_results) - len(y_true)
    if parse_errors > 0:
        print(f"⚠️ Warning: Ignored {parse_errors} sentences due to JSON Parse Errors.")
        
    acc = accuracy_score(y_true, y_pred)
    print(f"Overall Accuracy: {acc * 100:.2f}%")
    print("\nDetailed Classification Report:")
    
    # Zero_division=0 prevents warnings if a class is completely missed
    print(classification_report(y_true, y_pred, zero_division=0))

# Run metrics for all three dimensions
print_metrics("Hate Presence Detection", "true_presence", "pred_presence")
print_metrics("Hate Speech Type Extraction", "true_type", "pred_type")
print_metrics("Target Category Extraction", "true_target", "pred_target")